# Mô hình đối chứng 1: `HOG + SVM` (Học máy truyền thống)

Sổ tay này trình bày cơ sở lý thuyết, cấu tạo chi tiết và đánh giá mô hình đối chứng **`HOG + SVM`** trên cùng bộ dữ liệu đa văn hóa 3 nguồn (**KDEF + RADIATE + FER-2013**, 41.269 ảnh sạch, chia nghiêm ngặt theo người `Subject-Disjoint`) với mô hình CNN chính (`KDEF_Stem2Ch_MS4_Symmetric`).

---

## 1. Trình bày rõ cấu tạo & nguyên lý của `HOG + SVM`

Mô hình đại diện cho hướng **học máy truyền thống** (đặc trưng do con người thiết kế thủ công + bộ phân loại tuyến tính), gồm 2 thành phần nối tiếp:

### 1.1. Đặc trưng HOG (*Histogram of Oriented Gradients* — Dalal & Triggs, CVPR 2005)
HOG mô tả hình thái khuôn mặt thông qua phân bố hướng của các cạnh sáng–tối (gradient cục bộ):
1. **Đầu vào chuẩn hóa:** Ảnh xám khuôn mặt đã cắt theo SSD + CLAHE kích thước `160 × 160`.
2. **Tính đạo hàm không gian (Gradient):** Với mỗi điểm ảnh $(x, y)$, tính đạo hàm theo phương ngang $G_x = I(x+1,y) - I(x-1,y)$ và phương dọc $G_y = I(x,y+1) - I(x,y-1)$:
   - Độ lớn cạnh: $M(x,y) = \sqrt{G_x^2 + G_y^2}$
   - Góc hướng cạnh (không dấu): $\theta(x,y) = \arctan2(G_y, G_x) \bmod 180^\circ$
3. **Chia ô (Cells) & Biểu đồ hướng (Histogram):** Ảnh `160 × 160` được chia thành lưới `10 × 10` ô vuông (mỗi ô `16 × 16` pixel). Trong mỗi ô, góc $[0^\circ, 180^\circ)$ được chia thành **9 khoảng hướng (bins)** (mỗi bin $20^\circ$) và cộng dồn độ lớn $M$ vào bin tương ứng.
4. **Chuẩn hóa theo khối (Block Normalization — L2-Hys):** Gom `2 × 2` ô thành một khối trượt bước 1 ô (`9 × 9 = 81` khối). Mỗi khối có `2 × 2 × 9 = 36` giá trị được chuẩn hóa theo chuẩn $L_2$ cắt ngưỡng (`L2-Hys`) để giảm nhạy cảm với độ sáng.
5. **Vector đặc trưng đầu ra:** Nối toàn bộ 81 khối thu được vector đặc trưng kích thước $81 \times 36 = \mathbf{2.916}$ chiều, sau đó chuẩn hóa về trung bình 0, độ lệch chuẩn 1 (`StandardScaler` khớp duy nhất trên tập `train`).

### 1.2. Máy vector hỗ trợ tuyến tính (`LinearSVC` — Cortes & Vapnik, 1995)
- **Nguyên lý:** Tìm các siêu phẳng phân cách tuyến tính $w_c^\top x + b_c = 0$ theo chiến lược *One-vs-Rest* cho 7 lớp cảm xúc sao cho tối đa hóa khoảng cách lề (*maximum margin*) giữa các lớp bằng hàm mục tiêu Hinge Loss có điều chuẩn $L_2$:
  $$\min_{w, b} \frac{1}{2}\|w\|_2^2 + C \sum_{i=1}^{N} w_{y_i} \max\big(0, 1 - y_i(w^\top x_i + b)\big)$$
- **Cấu hình tinh chỉnh trên tập `val`:** Tham số phạt $C = 0,0002$, gán trọng số lớp `class_weight` để xử lý mất cân bằng 7 cảm xúc, kết hợp **hiệu chuẩn xác suất Platt Sigmoid** (`CalibratedClassifierCV`) để tính độ đo **ROC-AUC**.
- **Quy mô:** $2.916 \times 7 + 7 = \mathbf{20.419}$ tham số (tệp `checkpoints/hog_svm_multicultural.pkl` dung lượng **0,10 MB**), tốc độ suy luận CPU **~0,43 ms/ảnh** (~2.302 FPS).

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.models.baselines import HOGSVMClassifier, compute_multicultural_metrics, load_multicultural_split_cache

ckpt_path = REPO_ROOT / 'checkpoints' / 'hog_svm_multicultural.pkl'
print('Checkpoint:', ckpt_path, '| Exists:', ckpt_path.exists())

In [ ]:
# Đánh giá trên tập kiểm định (val) để xác minh không rò rỉ dữ liệu
val_imgs, val_labels, val_df = load_multicultural_split_cache('val', img_size=160)
clf = HOGSVMClassifier.load(ckpt_path)
val_probs = clf.predict_proba(val_imgs)
val_preds = val_probs.argmax(axis=1)
val_metrics = compute_multicultural_metrics(val_labels, val_preds, val_probs, meta_df=val_df)
print(f"[HOG + SVM | Val] Acc={val_metrics['accuracy']*100:.2f}% | Macro-F1={val_metrics['macro_f1']*100:.2f}% | ROC-AUC={val_metrics['roc_auc_ovr_macro']:.4f}")
for src, sm in val_metrics['per_source'].items():
    print(f"  - {src}: Acc={sm['accuracy']*100:.2f}% | Macro-F1={sm['macro_f1']*100:.2f}%")

In [ ]:
# Đọc kết quả so sánh chính thức trên tập test khóa (sinh bởi scripts/so_sanh_mo_hinh.py)
comp_path = REPO_ROOT / 'results' / 'test_comparison.json'
if comp_path.exists():
    comp = json.loads(comp_path.read_text(encoding='utf-8'))
    m = comp['models']['hog_svm']
    mt = m['metrics']
    print(f"[HOG + SVM | Locked Test n={mt['n_samples']}] Params={m['n_parameters']:,} | Size={m['checkpoint_mb']:.2f} MB")
    print(f"  Acc={mt['accuracy']*100:.2f}% | Macro-F1={mt['macro_f1']*100:.2f}% | Weighted-F1={mt['weighted_f1']*100:.2f}% | ROC-AUC={mt['roc_auc_ovr_macro']:.4f}")
    print(f"  CPU Latency: {m['latency_cpu']['ms_per_image']:.2f} ms/img ({m['latency_cpu']['fps']:.0f} FPS)")